# Step 4 — Stage-Conditioned WGAN-GP (Contribution C1)

**RESS 2025 — GAN-Conformal-RUL**

Trains one stage-conditioned WGAN-GP per fold to synthesise near-failure windows, then
assesses whether the synthetic windows are realistic **before** they are used downstream.

| | |
|---|---|
| **Loss** | WGAN-GP (resists mode collapse — critical for C1) |
| **Architecture** | Hybrid Conv-BiLSTM generator + critic (~0.9M params) |
| **Conditioning** | one-hot degradation stage → generator can be asked for near-failure |
| **Trained on** | each fold's **training windows only** (never cal/test → protects C3) |
| **Saved** | generator checkpoint per fold → `checkpoints/` (regenerate windows on demand) |

**Quality gates (a synthetic set must pass these to be trusted):**
1. Wasserstein distance converges (training curve flattens)
2. A fresh critic cannot separate real vs synthetic near-failure (accuracy → ~50%)
3. t-SNE: synthetic S3 windows overlap the real S3 cloud, not off in their own region
4. Feature-wise distributions match (real vs synthetic per-feature means/spreads)
5. No mode collapse (synthetic windows are diverse, not near-duplicates)

## 1. Setup

In [1]:
import os, sys, shutil
os.chdir('/content')

REPO_PATH = '/content/RESS_2025_GAN_Conformal_RUL'
if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH)
!git clone https://github.com/f-khadija-benzine/RESS_2025_GAN_Conformal_RUL.git {REPO_PATH}

os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, f'{REPO_PATH}/src')

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = f'{REPO_PATH}/figures'
CKPT_DIR = f'{REPO_PATH}/checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

import torch
print('Setup OK | CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

Cloning into '/content/RESS_2025_GAN_Conformal_RUL'...
remote: Enumerating objects: 249, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 249 (delta 31), reused 5 (delta 5), pack-reused 201 (from 1)
Receiving objects: 100% (249/249), 6.76 MiB | 11.15 MiB/s, done.
Resolving deltas: 100% (124/124), done.
Mounted at /content/drive
Setup OK | CUDA: True | Tesla T4


In [2]:
import numpy as np
import matplotlib.pyplot as plt

from data_loader import XJTUSYLoader
from health_indicator_v3 import HealthIndicatorPipeline
from windowing import prepare_all_folds, build_folds
from gan import StageGAN, GANConfig

# Stage color code (consistent across every figure)
STAGE_COLORS = {0: '#2ca02c', 1: '#ff9800', 2: '#e34948'}   # 0-indexed here
STAGE_NAMES  = {0: 'Healthy', 1: 'Early Deg.', 2: 'Near-Failure'}
TARGET = 2   # 0-indexed near-failure stage (S3)

[health_indicator] v2.0-guarded-fpt loaded  (FPT: 5 consecutive x max(mu+3sigma, mu*1.10))


## 2. Rebuild data (Steps 2–3)

In [3]:
CANDIDATES = ['/content/drive/MyDrive/XJTU-SY',
              '/content/drive/MyDrive/XJTU-SY_Bearing_Datasets',
              '/content/drive/MyDrive/data/XJTU-SY']
DATA_ROOT = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert DATA_ROOT, f'XJTU-SY not found in {CANDIDATES}'

all_data = XJTUSYLoader(DATA_ROOT).load_all()
pipeline = HealthIndicatorPipeline(fpt_consecutive=5, fpt_min_relative_rise=0.20)
results = pipeline.process_all(all_data, verbose=False)

fold_data = prepare_all_folds(results, verbose=False)
folds = build_folds(results)
print(f'{len(fold_data)} folds prepared.')
for d in fold_data:
    n_s3 = int((d['y_stage_train'] == TARGET).sum())
    print(f"  Fold {d['fold']}: {len(d['X_train'])} train windows, "
          f"{n_s3} near-failure ({100*n_s3/len(d['X_train']):.1f}%)")


=== Condition 1: 35.0 Hz, 12.0 kN ===
Loading Bearing1_1 (1-1): 123 recordings, failure mode: Outer race
  -> Shape: (123, 32768, 2), Memory: 32.2 MB
Loading Bearing1_2 (1-2): 161 recordings, failure mode: Outer race
  -> Shape: (161, 32768, 2), Memory: 42.2 MB
Loading Bearing1_3 (1-3): 158 recordings, failure mode: Outer race
  -> Shape: (158, 32768, 2), Memory: 41.4 MB
Loading Bearing1_4 (1-4): 122 recordings, failure mode: Cage
  -> Shape: (122, 32768, 2), Memory: 32.0 MB
Loading Bearing1_5 (1-5): 52 recordings, failure mode: Outer race + Ball
  -> Shape: (52, 32768, 2), Memory: 13.6 MB

=== Condition 2: 37.5 Hz, 11.0 kN ===
Loading Bearing2_1 (2-1): 491 recordings, failure mode: Inner race
  -> Shape: (491, 32768, 2), Memory: 128.7 MB
Loading Bearing2_2 (2-2): 161 recordings, failure mode: Outer race
  -> Shape: (161, 32768, 2), Memory: 42.2 MB
Loading Bearing2_3 (2-3): 533 recordings, failure mode: Cage
  -> Shape: (533, 32768, 2), Memory: 139.7 MB
Loading Bearing2_4 (2-4): 42 re

## 3. Train one GAN per fold

Each GAN sees only its fold's training windows. Checkpoints and loss history are saved to
`checkpoints/`. Set `EPOCHS` lower (e.g. 50) for a quick smoke test first.

In [4]:
EPOCHS = 50         # drop to 50 for a quick test run
AUGMENT_RATIO = 1.0   # ablation knob (saved with checkpoint; not used until Step 6)

gans = {}
for d in fold_data:
    k = d['fold']
    print(f"\n{'='*60}\nFOLD {k} — training WGAN-GP\n{'='*60}")
    cfg = GANConfig(epochs=EPOCHS, augment_ratio=AUGMENT_RATIO, target_stage=3)
    gan = StageGAN(cfg)
    gan.fit(d['X_train'], d['y_stage_train'], verbose=True)
    gans[k] = gan

    ckpt = {
        'generator': gan.G.state_dict(),
        'critic': gan.D.state_dict(),
        'config': cfg.__dict__,
        'history': gan.history,
        'fold': k,
    }
    #path = f'{CKPT_DIR}/gan_fold{k}.pt'
    #torch.save(ckpt, path)
    #print(f'  saved -> {path}')


FOLD 1 — training WGAN-GP
[StageGAN] Generator 532,244 params | Critic 361,857 params | device=cuda


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


  epoch    0 | D   -2.669 | G    1.385 | W-dist   3.5750
  epoch   20 | D  -14.560 | G   22.236 | W-dist  17.5452
  epoch   40 | D   -9.284 | G   28.799 | W-dist  11.8071
  epoch   49 | D   -9.095 | G   30.521 | W-dist  11.4396

FOLD 2 — training WGAN-GP
[StageGAN] Generator 532,244 params | Critic 361,857 params | device=cuda
  epoch    0 | D   -2.490 | G    1.467 | W-dist   3.3074
  epoch   20 | D   -7.531 | G   27.257 | W-dist   8.8984


KeyboardInterrupt: 

In [5]:
g = gans.get(1)  # may not exist if it errored before saving
# if not, grab the live one — re-run just enough to have `gan` in scope
import torch, numpy as np
samp = gan.sample(64, 2)   # 64 synthetic near-failure windows
print("synthetic shape:", samp.shape)
print("value range:", samp.min(), "to", samp.max())
print("per-window std (diversity):", samp.reshape(64,-1).std(1).mean())
print("real range for comparison:",
      fold_data[0]['X_train'].min(), "to", fold_data[0]['X_train'].max())

synthetic shape: (64, 32, 20)
value range: -2.0777254 to 4.634227
per-window std (diversity): 0.61250895
real range for comparison: -8.159209 to 228.7811


## 4. Quality gate 1 — Wasserstein convergence

The Wasserstein distance (critic's real−fake gap) should fall and flatten. A curve still
descending steeply at the end means undertraining; wild oscillation means instability.

In [ ]:
fig, axes = plt.subplots(1, len(gans), figsize=(4*len(gans), 3.2), sharey=True)
if len(gans) == 1: axes = [axes]
for ax, (k, gan) in zip(axes, gans.items()):
    ax.plot(gan.history['wasserstein'], color='#1f77b4', lw=1)
    ax.axhline(0, color='gray', ls=':', lw=0.8)
    ax.set_title(f'Fold {k}'); ax.set_xlabel('epoch')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Wasserstein distance')
plt.suptitle('WGAN-GP convergence (should fall and flatten)', y=1.03)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/04_wasserstein_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Quality gate 2 — a fresh critic can't tell real from synthetic

Train a small independent classifier to separate real near-failure windows from synthetic
ones. If the synthetic data is realistic, it should score **~50%** (a coin flip). High
accuracy means the synthetic windows are distinguishable — a red flag.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

print('Real-vs-synthetic separability (near-failure windows):')
print('  ~50% = indistinguishable (good) | high = distinguishable (bad)\n')
for k, gan in gans.items():
    d = next(x for x in fold_data if x['fold'] == k)
    real = d['X_train'][d['y_stage_train'] == TARGET]
    n = len(real)
    syn = gan.sample(n, TARGET)
    X = np.concatenate([real, syn]).reshape(2*n, -1)   # flatten 32x20 -> 640
    y = np.r_[np.zeros(n), np.ones(n)]
    acc = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=0),
                          X, y, cv=5, scoring='accuracy').mean()
    flag = 'good' if acc < 0.65 else ('borderline' if acc < 0.80 else 'POOR — distinguishable')
    print(f'  Fold {k}: {acc:.1%}  ({n} real vs {n} synthetic)  [{flag}]')

## 6. Quality gate 3 — t-SNE overlap

Project real and synthetic near-failure windows into 2-D. Synthetic points should sit
**within** the real cloud. A separate synthetic cluster means the GAN learned a different
distribution; a single tight synthetic blob means mode collapse.

In [ ]:
from sklearn.manifold import TSNE

fig, axes = plt.subplots(1, len(gans), figsize=(4*len(gans), 3.6))
if len(gans) == 1: axes = [axes]
for ax, (k, gan) in zip(axes, gans.items()):
    d = next(x for x in fold_data if x['fold'] == k)
    real = d['X_train'][d['y_stage_train'] == TARGET]
    n = min(len(real), 300)
    real = real[np.random.choice(len(real), n, replace=False)]
    syn = gan.sample(n, TARGET)
    Z = np.concatenate([real, syn]).reshape(2*n, -1)
    emb = TSNE(n_components=2, perplexity=30, random_state=0,
               init='pca', learning_rate='auto').fit_transform(Z)
    ax.scatter(emb[:n,0], emb[:n,1], s=8, alpha=0.6, color='#2166d4', label='real S3')
    ax.scatter(emb[n:,0], emb[n:,1], s=8, alpha=0.6, color='#e34948', label='synthetic S3')
    ax.set_title(f'Fold {k}'); ax.set_xticks([]); ax.set_yticks([])
    if k == list(gans)[0]: ax.legend(fontsize=8)
plt.suptitle('t-SNE — synthetic near-failure should overlap real', y=1.03)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/04_tsne_real_vs_synthetic.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Quality gate 4 — feature-wise distribution match

For a representative fold, compare the per-feature value distributions of real vs synthetic
near-failure windows. They should overlap. Systematic offset means the GAN mis-scaled a feature.

In [ ]:
FEATURE_NAMES = results[list(results)[0]]['feature_names']
k = list(gans)[0]
gan = gans[k]
d = next(x for x in fold_data if x['fold'] == k)
real = d['X_train'][d['y_stage_train'] == TARGET]
syn = gan.sample(len(real), TARGET)

fig, axes = plt.subplots(4, 5, figsize=(16, 10))
for i, ax in enumerate(axes.flat):
    ax.hist(real[:,:,i].ravel(), bins=40, alpha=0.5, density=True,
            color='#2166d4', label='real')
    ax.hist(syn[:,:,i].ravel(), bins=40, alpha=0.5, density=True,
            color='#e34948', label='synthetic')
    ax.set_title(FEATURE_NAMES[i], fontsize=8)
    ax.tick_params(labelsize=6)
    if i == 0: ax.legend(fontsize=7)
plt.suptitle(f'Fold {k} — per-feature distributions, real vs synthetic near-failure', y=1.01)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/04_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Quality gate 5 — diversity (no mode collapse)

Mode collapse = the generator emits near-identical windows. Measure the mean pairwise
distance among synthetic windows and compare it to that among real windows. Synthetic
diversity far below real is the collapse signature.

In [ ]:
from scipy.spatial.distance import pdist

print('Diversity — mean pairwise distance (synthetic / real ratio):')
print('  ~1.0 = matched diversity | << 1.0 = mode collapse\n')
for k, gan in gans.items():
    d = next(x for x in fold_data if x['fold'] == k)
    real = d['X_train'][d['y_stage_train'] == TARGET]
    n = min(len(real), 200)
    real = real[np.random.choice(len(real), n, replace=False)].reshape(n, -1)
    syn = gan.sample(n, TARGET).reshape(n, -1)
    dr, ds = pdist(real).mean(), pdist(syn).mean()
    ratio = ds / dr
    flag = 'good' if ratio > 0.7 else ('borderline' if ratio > 0.5 else 'COLLAPSE risk')
    print(f'  Fold {k}: real {dr:.3f} | synthetic {ds:.3f} | ratio {ratio:.2f}  [{flag}]')

## 9. Visual sanity — real vs synthetic trajectories

Plot RMS (feature 0) over the 32-step window for a handful of real and synthetic
near-failure samples. Synthetic should look like plausible degradation, not noise.

In [ ]:
k = list(gans)[0]
gan = gans[k]
d = next(x for x in fold_data if x['fold'] == k)
real = d['X_train'][d['y_stage_train'] == TARGET]
syn = gan.sample(8, TARGET)
real_s = real[np.random.choice(len(real), 8, replace=False)]

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for w in real_s:
    axes[0].plot(w[:, 0], color='#2166d4', alpha=0.6, lw=1)
for w in syn:
    axes[1].plot(w[:, 0], color='#e34948', alpha=0.6, lw=1)
axes[0].set_title(f'Fold {k} — real near-failure (horiz RMS)')
axes[1].set_title(f'Fold {k} — synthetic near-failure (horiz RMS)')
for ax in axes: ax.set_xlabel('time step within window'); ax.grid(alpha=0.3)
axes[0].set_ylabel('scaled RMS')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/04_real_vs_synthetic_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

Checkpoints saved to `checkpoints/gan_fold{k}.pt` — commit these (final runs only).
The synthetic windows themselves are **not** saved; they are regenerated in Step 6 from the
generator weights via `gan.augment(...)`, at whatever `augment_ratio` each ablation row needs.

**Decision gate before Step 6:** proceed only if the five quality gates pass —
Wasserstein converged, real-vs-synthetic separability near 50%, t-SNE overlap, feature
distributions matched, diversity ratio near 1. If a fold fails, its synthetic near-failure
data is not trustworthy and augmentation there would inject noise rather than signal.